# Dependent designs — common-shock, copula, MRV

Walks through the manuscript's three dependent-extension paths:

1. **Latent-shock CdMC** under a common-shock model (Theorem `thm:latent-shock-tail`).
2. **Copula-kernel CdMC** with a Clayton copula in dimension 3.
3. **Spectral CdMC** under MRV with a Dirichlet angular distribution.

Each estimator returns a `CdMCResult` with `mu_hat`, `variance`,
Bernstein CI, and estimator-specific diagnostics in `extra`.

In [ ]:
import numpy as np

## §4 latent-shock CdMC

A factor model $X = BZ + E$ with two independent Pareto shocks and three
Lomax idiosyncratic terms. The latent-shock CdMC reduces the problem
to the shock basis $q = B^\top a$.

In [ ]:
from factortail.cdmc import latent_shock_cdmc
from factortail.utils.tails import LomaxTail, ParetoTail

B = np.array([[1.0, 0.0], [0.5, 0.5]])  # two shocks, two observed
shocks = [ParetoTail(alpha=2.0, scale=1.0)] * 2
idio = [LomaxTail(alpha=2.0, scale=0.1)] * 2
a = np.array([1.0, 1.0])

res_latent = latent_shock_cdmc(
    B=B, exposure=a, shocks=shocks, idiosyncratic=idio,
    x=15.0, n=10_000, seed=0,
)
print(f'mu_hat        = {res_latent.mu_hat:.4e}')
print(f'active shocks = {res_latent.extra["active_shocks"]}')
print(f'envelope      = {res_latent.extra["envelope"]:.4e}')

## §4 copula-kernel CdMC

Three Pareto margins coupled by a Clayton copula. The closed-form
Archimedean conditional CDF $F_{i|-i}(t|u_{-i})$ in arbitrary
dimension is in `factortail.copula.ClaytonCopula`; the kernel wrapper
is `build_copula_kernel_batched`.

In [ ]:
from factortail.cdmc import (
    build_copula_kernel_batched,
    build_copula_sampler,
    dependent_cdmc,
)
from factortail.copula import ClaytonCopula

marginals = [ParetoTail(alpha=2.0, scale=1.0)] * 3
cop = ClaytonCopula(theta=2.0, d=3)
sampler = build_copula_sampler(cop, marginals)
kernel_batch = build_copula_kernel_batched(cop, marginals)

res_cop = dependent_cdmc(
    sampler=sampler, kernel_batch=kernel_batch,
    x=10.0, n=2000, seed=42,
)
print(f'mu_hat      = {res_cop.mu_hat:.4e}')
print(f'rel SE      = {res_cop.rel_sd:.4f}')
print(f'kernel_kind = {res_cop.extra["kernel_kind"]}  (batched is ~7.5x faster than scalar)')

## §5 spectral CdMC under MRV

Radial-angular MRV with an exact Pareto radial and Dirichlet(2,2,2)
angular mass on the 2-simplex. The estimator integrates the radial
survival against the loss functional $\ell(\theta) = a^\top \theta$.

In [ ]:
from factortail.cdmc import spectral_cdmc
from factortail.dgp import RadialAngularMRV

dgp = RadialAngularMRV(
    alpha=2.0,
    angular_kind='dirichlet',
    angular_params={'concentration': [2.0, 2.0, 2.0]},
    dim=3,
)
exposure = np.array([1.0, 2.0, 0.5])

res_spec = spectral_cdmc(
    angle_sampler=lambda n, r: dgp.sample_angles(n, r),
    radial=dgp.radial,
    exposure=exposure,
    x=10.0,
    n=10_000,
    seed=1,
)
print(f'mu_hat              = {res_spec.mu_hat:.4e}')
print(f'spectral constant   = {res_spec.extra["spectral_constant"]:.4e}')
print(f'rel SE              = {res_spec.rel_sd:.4f}')

## Side-by-side

Three estimators, three dependence structures, one common
`CdMCResult` interface.

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'estimator': ['latent_shock', 'copula_clayton', 'spectral_mrv'],
    'mu_hat':    [res_latent.mu_hat, res_cop.mu_hat, res_spec.mu_hat],
    'rel_sd':    [res_latent.rel_sd, res_cop.rel_sd, res_spec.rel_sd],
    'runtime_s': [res_latent.runtime_seconds, res_cop.runtime_seconds, res_spec.runtime_seconds],
})
summary

## Where to go next

- **[Real-data pipeline](03_real_data_pipeline.ipynb)** — rolling
  VaR/ES on Fama–French.
- **[Results gallery](../results.md)** — every generated figure with
  its diagnostic interpretation.
- **[Diagnostics](../diagnostics.md)** — Hill / Pickands / POT,
  $\chi$/$\bar\chi$/$\eta$, bootstrap bands on the spectral measure.